In [1]:
from requests import Session
import requests
import json
import urllib3
from bs4 import BeautifulSoup
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
import os
num_cores = os.cpu_count()
num_workers = num_cores * 2

In [ ]:
with Session() as session:
    session.head(f'https://www.tender.gov.mn/mn/result/detail/{tender_id}', verify = False)
    response = session.post(
        url = sub_tender_url,
        headers={
            'Referer': f'https://www.tender.gov.mn/mn/result/detail/{tender_id}',
            'Origin': 'https://www.tender.gov.mn',
            'host': 'www.tender.gov.mn',
            'accept': 'application/json, text/javascript, */*; q=0.01'
        },
        data = payload
    )

In [11]:
response = requests.get("https://api.tender.gov.mn/api/process/300/list?endBudget=10000000000000&limit=1000000&offset=1&order=openDate-&tenderYear=2025")

19899

# Concurrent CSV Writing in Python

There are several approaches to handle concurrent writing to CSV files:

1. **Thread-safe writing with locks** - Multiple threads write to the same file with synchronization
2. **Queue-based writing** - Workers add data to a queue, single writer consumes from queue
3. **Separate files + merge** - Each worker writes to separate files, then merge
4. **Concurrent.futures with thread pool** - Use ThreadPoolExecutor for parallel processing

In [ ]:
# Additional imports for concurrent CSV writing
import threading
import queue
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv
from threading import Lock

In [ ]:
# Approach 1: Thread-safe writing with locks
class ThreadSafeCSVWriter:
    def __init__(self, filename, fieldnames):
        self.filename = filename
        self.fieldnames = fieldnames
        self.lock = Lock()
        self._init_file()
    
    def _init_file(self):
        """Initialize CSV file with headers"""
        with open(self.filename, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=self.fieldnames)
            writer.writeheader()
    
    def write_row(self, row_data):
        """Thread-safe method to write a single row"""
        with self.lock:
            with open(self.filename, 'a', newline='', encoding='utf-8') as file:
                writer = csv.DictWriter(file, fieldnames=self.fieldnames)
                writer.writerow(row_data)

# Example usage of thread-safe writer
def worker_function(writer, worker_id, data_batch):
    """Simulated worker function that processes data and writes to CSV"""
    for i, item in enumerate(data_batch):
        # Simulate some processing time
        time.sleep(0.1)
        
        # Prepare row data
        row = {
            'worker_id': worker_id,
            'item_id': i,
            'data': item,
            'timestamp': time.time()
        }
        
        # Write to CSV in thread-safe manner
        writer.write_row(row)
        print(f"Worker {worker_id} wrote item {i}")

# Demo data
sample_data = [
    ['data1', 'data2', 'data3'] * 5,  # Worker 1 data
    ['dataA', 'dataB', 'dataC'] * 5,  # Worker 2 data
    ['dataX', 'dataY', 'dataZ'] * 5   # Worker 3 data
]

# Initialize thread-safe CSV writer
fieldnames = ['worker_id', 'item_id', 'data', 'timestamp']
csv_writer = ThreadSafeCSVWriter('concurrent_output.csv', fieldnames)

print("Starting thread-safe CSV writing demo...")

In [ ]:
# Run the thread-safe demo
threads = []
for i, data_batch in enumerate(sample_data):
    thread = threading.Thread(
        target=worker_function, 
        args=(csv_writer, f"Worker_{i+1}", data_batch)
    )
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

print("Thread-safe CSV writing completed!")

# Read and display the results
df_result = pd.read_csv('concurrent_output.csv')
print(f"\nTotal rows written: {len(df_result)}")
print("\nFirst 10 rows:")
print(df_result.head(10))

In [ ]:
# Approach 2: Queue-based writing (Producer-Consumer pattern)
class QueueBasedCSVWriter:
    def __init__(self, filename, fieldnames):
        self.filename = filename
        self.fieldnames = fieldnames
        self.data_queue = queue.Queue()
        self.stop_event = threading.Event()
        self.writer_thread = None
        
    def start_writer(self):
        """Start the background writer thread"""
        # Initialize CSV file
        with open(self.filename, 'w', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=self.fieldnames)
            writer.writeheader()
        
        self.writer_thread = threading.Thread(target=self._writer_worker)
        self.writer_thread.start()
    
    def _writer_worker(self):
        """Background thread that consumes from queue and writes to CSV"""
        with open(self.filename, 'a', newline='', encoding='utf-8') as file:
            writer = csv.DictWriter(file, fieldnames=self.fieldnames)
            
            while not self.stop_event.is_set() or not self.data_queue.empty():
                try:
                    # Get data from queue with timeout
                    row_data = self.data_queue.get(timeout=1)
                    writer.writerow(row_data)
                    file.flush()  # Ensure data is written immediately
                    self.data_queue.task_done()
                except queue.Empty:
                    continue
    
    def write_row(self, row_data):
        """Add row to queue for writing"""
        self.data_queue.put(row_data)
    
    def stop_and_wait(self):
        """Stop the writer and wait for all data to be written"""
        self.data_queue.join()  # Wait for all items to be processed
        self.stop_event.set()
        if self.writer_thread:
            self.writer_thread.join()

# Producer function for queue-based approach
def producer_function(queue_writer, producer_id, data_batch):
    """Producer that adds data to the queue"""
    for i, item in enumerate(data_batch):
        # Simulate some processing
        time.sleep(0.05)
        
        row = {
            'producer_id': producer_id,
            'item_id': i,
            'data': item,
            'timestamp': time.time()
        }
        
        queue_writer.write_row(row)
        print(f"Producer {producer_id} queued item {i}")

print("Queue-based CSV writing approach ready...")

In [ ]:
# Run the queue-based demo
fieldnames = ['producer_id', 'item_id', 'data', 'timestamp']
queue_writer = QueueBasedCSVWriter('queue_output.csv', fieldnames)

# Start the background writer
queue_writer.start_writer()

# Start multiple producers
producer_threads = []
for i, data_batch in enumerate(sample_data):
    thread = threading.Thread(
        target=producer_function,
        args=(queue_writer, f"Producer_{i+1}", data_batch)
    )
    producer_threads.append(thread)
    thread.start()

# Wait for all producers to finish
for thread in producer_threads:
    thread.join()

# Stop the writer and wait for all data to be written
queue_writer.stop_and_wait()

print("Queue-based CSV writing completed!")

# Read and display results
df_queue_result = pd.read_csv('queue_output.csv')
print(f"\nTotal rows written: {len(df_queue_result)}")
print("\nFirst 10 rows:")
print(df_queue_result.head(10))

In [ ]:
# Approach 3: Using ThreadPoolExecutor for concurrent processing
def process_and_collect_data(worker_id, data_batch):
    """Process data and return results to be written later"""
    results = []
    for i, item in enumerate(data_batch):
        # Simulate processing
        time.sleep(0.1)
        
        result = {
            'worker_id': worker_id,
            'item_id': i,
            'data': item,
            'processed_data': f"processed_{item}",
            'timestamp': time.time()
        }
        results.append(result)
        print(f"Worker {worker_id} processed item {i}")
    
    return results

def concurrent_csv_with_threadpool(data_batches, filename, max_workers=None):
    """Use ThreadPoolExecutor to process data concurrently, then write to CSV"""
    if max_workers is None:
        max_workers = min(len(data_batches), num_workers)
    
    all_results = []
    
    # Process data concurrently
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_worker = {
            executor.submit(process_and_collect_data, f"Worker_{i+1}", batch): i 
            for i, batch in enumerate(data_batches)
        }
        
        # Collect results as they complete
        for future in as_completed(future_to_worker):
            worker_idx = future_to_worker[future]
            try:
                results = future.result()
                all_results.extend(results)
                print(f"Completed worker {worker_idx + 1}")
            except Exception as exc:
                print(f"Worker {worker_idx + 1} generated an exception: {exc}")
    
    # Write all results to CSV at once
    if all_results:
        df = pd.DataFrame(all_results)
        df.to_csv(filename, index=False)
        print(f"Wrote {len(all_results)} rows to {filename}")
    
    return all_results

print("ThreadPoolExecutor approach ready...")

In [ ]:
# Run the ThreadPoolExecutor demo
print("Starting ThreadPoolExecutor demo...")
start_time = time.time()

results = concurrent_csv_with_threadpool(
    sample_data, 
    'threadpool_output.csv', 
    max_workers=3
)

end_time = time.time()
print(f"ThreadPoolExecutor completed in {end_time - start_time:.2f} seconds")

# Read and display results
df_threadpool = pd.read_csv('threadpool_output.csv')
print(f"\nTotal rows written: {len(df_threadpool)}")
print("\nFirst 10 rows:")
print(df_threadpool.head(10))

In [ ]:
# Real-world example: Concurrent processing of tender data
def fetch_and_process_tender_data(tender_ids_batch, writer, batch_id):
    """
    Fetch tender data for a batch of IDs and write to CSV concurrently
    This is a mock function - replace with your actual API calls
    """
    for tender_id in tender_ids_batch:
        try:
            # Simulate API call delay
            time.sleep(0.2)
            
            # Mock tender data - replace with actual API response processing
            tender_data = {
                'tender_id': tender_id,
                'batch_id': batch_id,
                'title': f'Tender {tender_id} Title',
                'budget': tender_id * 1000,
                'status': 'Active',
                'fetch_timestamp': time.time()
            }
            
            # Write to CSV using thread-safe writer
            writer.write_row(tender_data)
            print(f"Batch {batch_id}: Processed tender {tender_id}")
            
        except Exception as e:
            print(f"Error processing tender {tender_id}: {e}")

# Example usage with mock tender IDs
mock_tender_ids = list(range(1001, 1051))  # 50 tender IDs
batch_size = 10
tender_batches = [
    mock_tender_ids[i:i + batch_size] 
    for i in range(0, len(mock_tender_ids), batch_size)
]

# Set up CSV writer for tender data
tender_fieldnames = ['tender_id', 'batch_id', 'title', 'budget', 'status', 'fetch_timestamp']
tender_writer = ThreadSafeCSVWriter('tender_data_concurrent.csv', tender_fieldnames)

print(f"Processing {len(mock_tender_ids)} tenders in {len(tender_batches)} batches...")
print("Starting concurrent tender data processing...")

In [ ]:
# Process tender data concurrently
start_time = time.time()

threads = []
for i, batch in enumerate(tender_batches):
    thread = threading.Thread(
        target=fetch_and_process_tender_data,
        args=(batch, tender_writer, f"Batch_{i+1}")
    )
    threads.append(thread)
    thread.start()

# Wait for all threads to complete
for thread in threads:
    thread.join()

end_time = time.time()
print(f"\nConcurrent tender processing completed in {end_time - start_time:.2f} seconds")

# Read and analyze the results
df_tenders = pd.read_csv('tender_data_concurrent.csv')
print(f"\nTotal tenders processed: {len(df_tenders)}")
print(f"Batches processed: {df_tenders['batch_id'].nunique()}")
print("\nSample data:")
print(df_tenders.head(10))

# Summary statistics
print(f"\nTotal budget: ${df_tenders['budget'].sum():,}")
print(f"Average budget: ${df_tenders['budget'].mean():,.2f}")
print(f"Processing time range: {df_tenders['fetch_timestamp'].max() - df_tenders['fetch_timestamp'].min():.2f} seconds")

# Performance Comparison & Best Practices

## When to Use Each Approach:

### 1. **Thread-Safe Writing (with Locks)**
- ✅ **Best for**: Continuous data streams, real-time processing
- ✅ **Pros**: Simple, immediate writing, good for memory efficiency
- ❌ **Cons**: Lock contention can slow down with many threads
- **Use case**: Web scraping, API monitoring, real-time data collection

### 2. **Queue-Based Writing (Producer-Consumer)**
- ✅ **Best for**: High-throughput scenarios, batch processing
- ✅ **Pros**: Decouples producers from I/O, better performance with many writers
- ❌ **Cons**: More complex, uses more memory for queue
- **Use case**: Large-scale data processing, ETL pipelines

### 3. **ThreadPoolExecutor + Batch Write**
- ✅ **Best for**: Processing then writing, when you can collect all data first
- ✅ **Pros**: Simple, good performance, easier error handling
- ❌ **Cons**: Requires holding all data in memory
- **Use case**: Data transformation, analysis with final export

## Performance Tips:
- Use `max_workers = min(32, (os.cpu_count() or 1) + 4)` for I/O bound tasks
- For CPU-bound tasks, use `ProcessPoolExecutor` instead
- Consider using `pandas.DataFrame.to_csv()` for large datasets
- Use appropriate buffer sizes and flush strategies